# The biharmonic plate

A thin elastic plate bends under a transverse load. Its vertical deflection is described by a scalar function $w$.

## Plate model

After scaling the material constants, the plate equation is

$$
\Delta^2 w=f \qquad\text{in }\Omega.
$$

A clamped edge cannot move or rotate, so

$$
w=0,\qquad \frac{\partial w}{\partial n}=0
\qquad\text{on }\partial\Omega.
$$

The corresponding bending energy has the schematic form

$$
J(w)=\frac12\int_\Omega |\Delta w|^2\,dx-\int_\Omega fw\,dx.
$$

This is a fourth-order problem: the equation contains four derivatives of $w$. Its weak formulation is: Find $w\in H_0^2(\Omega)$ such that for all $v\in H_0^2(\Omega)$

$$
\int_{\Omega}\nabla^2 w:\nabla^2 v\,dx = \int_{\Omega} f\,v\,dx.
$$

In [ ]:
from netgen.occ import OCCGeometry, Rectangle
from ngsolve import Mesh, H1, GridFunction, BilinearForm, LinearForm, CF, Grad, dx
from ngsolve.webgui import Draw

plate = Rectangle(1, 1).Face()
plate.edges.name = "clamped"
mesh = Mesh(OCCGeometry(plate, dim=2).GenerateMesh(maxh=0.12))
Draw(mesh);

In [ ]:
# Numerical solver - this mixed formulation is only a black box today
W = H1(mesh, order=3, dirichlet="clamped")
S = H1(mesh, order=3)
space = W * S
(w, sigma), (v, tau) = space.TnT()

system = BilinearForm(space)
system += (sigma*tau + Grad(tau)*Grad(w) + Grad(sigma)*Grad(v) - 1e-8*w*v) * dx
load = LinearForm(space)
load += 50 * v * dx
system.Assemble()
load.Assemble()

solution = GridFunction(space)
solution.vec.data = system.mat.Inverse(space.FreeDofs(), inverse="sparsecholesky") * load.vec
deflection, auxiliary_variable = solution.components

In [ ]:
plate_deformation = CF((0, 0, 15*deflection))
Draw(deflection, mesh, "plate deflection", deformation=plate_deformation);

## Observe

- How do the clamped boundary conditions appear in the plot?
- How would you expect a longer or thinner plate to behave?


[← Minimal surfaces](02_minimal_surface_and_membrane.ipynb) · [Lecture overview](index.ipynb) · [Next: linear elasticity →](04_linear_elasticity.ipynb)